# 🍄 Boscdex Classifier Training on Google Colab (Free T4 GPU)

This notebook fine-tunes the 47-species on-device mushroom classifier for **Boscdex** using **MobileNetV2** and **Transfer Learning**.

### 🚀 Highlights:
1. **1-Click Google Drive Ingestion & Contamination Purge:** Automatically extracts `boscdex_dataset.zip` and purges any legacy `inat_*.jpg` files from earlier runs.
2. **100% Verified Fungi iNaturalist Top-Up (60+ images/species):** Uses verified Kingdom `Fungi` taxon IDs (`iconic_taxa="Fungi"`) across all 47 Catalan species.
3. **Stratified Split, Safety-Weighted Loss & Label Smoothing:** Deterministic per-species 85/15 stratified split, `1.20x` safety weight boost on lethal/toxic taxa, `CategoricalCrossentropy(label_smoothing=0.10)`, `AdamW`, and frozen `BatchNormalization` fine-tuning.
4. **Decoupled Edge-ML Architecture:** Exports both the **TFLite feature extractor** (`mushroom_classifier.tflite`) and the **binary classification head** (`mushroom_classifier_head.bin`), 100% compatible with the Boscdex Flutter app!

### ⚡ Quick Start:
1. Ensure runtime is set to **T4 GPU** (`Runtime > Change runtime type > T4 GPU`).
2. Click **Runtime > Run all** (or run each cell sequentially).
3. When finished, your browser will automatically download the 3 updated model files!

In [1]:
# Step 1: Verify Hardware & GPU Allocation
import sys
import platform
import shutil
import tensorflow as tf

print(f"Python Version:      {sys.version.split()[0]}")
print(f"Platform:            {platform.platform()}")
print(f"TensorFlow Version:  {tf.__version__}")

gpus = tf.config.list_physical_devices("GPU")
if gpus:
    print(f"✅ GPU Acceleration Active: {gpus[0].name}")
    !nvidia-smi --query-gpu=name,memory.total,memory.free --format=csv,noheader
else:
    print("⚠️ No GPU detected. For 10x faster training, select: Runtime > Change runtime type > T4 GPU.")


Python Version:      3.13.15
Platform:            Linux-6.6.122+-x86_64-with-glibc2.39
TensorFlow Version:  2.21.0
✅ GPU Acceleration Active: /physical_device:GPU:0
Tesla T4, 15360 MiB, 14910 MiB


In [2]:
# Step 2: Install Required Libraries
!pip install -q pyinaturalist requests pillow tqdm


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 141.1/141.1 kB 12.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.8/70.8 kB 8.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.8/74.8 kB 9.0 MB/s eta 0:00:00


In [3]:
# Step 3: Verified 47-Species Catalog & Directory Structure
import os
from pathlib import Path

DATASET_DIR = Path("./dataset")
SPLIT_DIR = Path("./dataset_split")
EXPORT_DIR = Path("./export")
DATASET_DIR.mkdir(parents=True, exist_ok=True)
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

# 47 Catalan Fungi Taxa in Boscdex with 100% verified active iNaturalist Fungi taxon_ids
SPECIES_CATALOG = [
    ("lactarius_deliciosus", 155197, "Rovelló / Pinetell"),
    ("boletus_edulis", 48701, "Cep"),
    ("cantharellus_cibarius", 47347, "Rossinyol"),
    ("amanita_caesarea", 204588, "Ou de reig"),
    ("craterellus_cornucopioides", 48607, "Trompeta de la mort"),
    ("macrolepiota_procera", 63401, "Apagallums"),
    ("tricholoma_terreum", 341797, "Fredolic"),
    ("calocybe_gambosa", 450829, "Moixernó"),
    ("pleurotus_ostreatus", 48494, "Gírgola"),
    ("hydnum_repandum", 48641, "Llengua de bou"),
    ("hygrophorus_latitabundus", 1440332, "Llenega negra"),
    ("craterellus_lutescens", 514218, "Camagroc"),
    ("suillus_luteus", 53488, "Llenegot"),
    ("russula_cyanoxantha", 194437, "Llora verda"),
    ("amanita_muscaria", 48715, "Reig bord"),
    ("omphalotus_olearius", 64021, "Gírgola d'olivera"),
    ("entoloma_sinuatum", 510199, "Pèrfid"),
    ("amanita_phalloides", 52135, "Farinera borda"),
    ("amanita_verna", 58684, "Cicuta blanca"),
    ("cortinarius_orellanus", 337849, "Cortinari de muntanya"),
    ("lactarius_sanguifluus", 332501, "Rovelló de sang / Pinetell de sang"),
    ("chroogomphus_rutilus", 194275, "Cama de perdiu"),
    ("marasmius_oreades", 118240, "Carrereta / Cama-sec"),
    ("lactarius_chrysorrheus", 194433, "Lleterola de llet groga / Fals rovelló"),
    ("lepista_nuda", 1525548, "Pé blau / Pua blava"),
    ("rubroboletus_satanas", 544554, "Mataparent de Satanàs"),
    ("russula_virescens", 125720, "Palometa / Llora pigada"),
    ("hygrophorus_eburneus", 67331, "Llenega blanca / Mocosa blanca"),
    ("tricholoma_pardinum", 335206, "Fredolic bord / Griset bord"),
    ("morchella_esculenta", 58682, "Múrgola / Rabassola"),
    ("agaricus_campestris", 143563, "Xampinyó silvestre / Guisana"),
    ("agaricus_xanthodermus", 118394, "Xampinyó groguenc / Xampinyó pudent"),
    ("psathyrella_candolleana", 1179954, "Candol·lea / Psatirel·la de Candolle"),
    ("suillus_granulatus", 179065, "Molleric clapat / Pinetell de pi blanc"),
    ("hygrophorus_russula", 206118, "Carlet / Escarlet"),
    ("boletus_aereus", 333772, "Cep negre / Sureny fosc"),
    ("coprinus_comatus", 47392, "Bolet de tinta / Braguer de vaca"),
    ("lactarius_torminosus", 351313, "Cabra / Rovelló bord"),
    ("amanita_pantherina", 48418, "Pantera / Reig bord pigat"),
    ("gyromitra_esculenta", 85120, "Bolet de greix / Falsa múrgola"),
    ("hygrophorus_personii", 1440333, "Llenega d'alzina"),
    ("pleurotus_eryngii", 205516, "Gírgola de panical"),
    ("lactarius_semisanguifluus", 775629, "Pinetell de sang"),
    ("suillus_bellinii", 499617, "Pinet de pi bord / Molleric de Bellini"),
    ("tricholoma_equestre", 118215, "Groguet / Verderol"),
    ("amanita_virosa", 58685, "Àngel de la mort / Farinera pudent"),
    ("galerina_marginata", 154735, "Galerina mortal"),
]

for s_id, _, _ in SPECIES_CATALOG:
    (DATASET_DIR / s_id).mkdir(parents=True, exist_ok=True)

print(f"✓ Configured {len(SPECIES_CATALOG)} verified fungal species directories in {DATASET_DIR.resolve()}")


✓ Configured 47 verified fungal species directories in /content/dataset


In [4]:
# Step 4: Dataset Ingestion & Legacy Contamination Purge
import zipfile
import shutil

# 1. Mount Google Drive if running in Google Colab
try:
    from google.colab import drive
    drive.mount("/content/drive")
    print("✅ Google Drive mounted successfully.")
except Exception as e:
    print(f"ℹ️ Google Drive mount note: {e}")

# 2. Check candidate locations for boscdex_dataset.zip
candidate_paths = [
    Path("/content/drive/MyDrive/boscdex_dataset.zip"),
    Path("/content/drive/MyDrive/boscdex/boscdex_dataset.zip"),
    Path("./boscdex_dataset.zip"),
]

found_archive = next((p for p in candidate_paths if p.exists()), None)

if found_archive:
    print(f"📦 Found dataset archive: {found_archive} ({found_archive.stat().st_size / (1024*1024):.1f} MB)")
    print("Extracting images into ./dataset/...")
    with zipfile.ZipFile(found_archive, "r") as zf:
        zf.extractall(DATASET_DIR)
    print("✅ Extraction complete!")
else:
    print("⚠️ No boscdex_dataset.zip found in Google Drive root or current working directory.")
    print("   Upload boscdex_dataset.zip to Google Drive (MyDrive/) or click the folder icon on the left to upload it.")

# 3. Purge any legacy inat_*.jpg files from previous runs so only verified Fungi are downloaded in Step 5
purged_count = 0
for old_inat in DATASET_DIR.rglob("inat_*.*"):
    try:
        old_inat.unlink()
        purged_count += 1
    except Exception:
        pass
if purged_count > 0:
    print(f"🧹 Purged {purged_count} legacy inat_*.jpg files to ensure 100% pure fungal dataset.")

# 4. Print clean field photo census
print()
print("📊 Clean Field Photo Census by Species:")
total_images = 0
for s_id, _, name in SPECIES_CATALOG:
    folder = DATASET_DIR / s_id
    count = len([p for p in folder.glob("*.*") if p.suffix.lower() in [".jpg", ".jpeg", ".png"]]) if folder.exists() else 0
    total_images += count
    print(f"  • {s_id.ljust(28)}: {count:3d} images ({name})")
print(f"Total curated field images available: {total_images}")


Mounted at /content/drive
✅ Google Drive mounted successfully.
📦 Found dataset archive: /content/drive/MyDrive/boscdex/boscdex_dataset.zip (155.3 MB)
Extracting images into ./dataset/...
✅ Extraction complete!

📊 Clean Field Photo Census by Species:
  • lactarius_deliciosus        :  30 images (Rovelló / Pinetell)
  • boletus_edulis              :  14 images (Cep)
  • cantharellus_cibarius       :   7 images (Rossinyol)
  • amanita_caesarea            :  14 images (Ou de reig)
  • craterellus_cornucopioides  :   6 images (Trompeta de la mort)
  • macrolepiota_procera        :   7 images (Apagallums)
  • tricholoma_terreum          :  11 images (Fredolic)
  • calocybe_gambosa            :   5 images (Moixernó)
  • pleurotus_ostreatus         :  15 images (Gírgola)
  • hydnum_repandum             :  13 images (Llengua de bou)
  • hygrophorus_latitabundus    :  22 images (Llenega negra)
  • craterellus_lutescens       :  10 images (Camagroc)
  • suillus_luteus              :   7 images (L

In [5]:
# Step 5: Verified Fungi iNaturalist Top-Up (60+ photos per species)
import time
import io
from pathlib import Path
from PIL import Image
import requests
from tqdm.notebook import tqdm
from pyinaturalist import get_observations

# Target minimum images per species:
# 60 images per species x 47 classes = ~2,820 balanced fungal training photos
MIN_IMAGES_PER_SPECIES = 60

PLACE_CATALONIA = 7161
PLACE_SPAIN = 6774
PLACE_EUROPE = 97391

HEADERS = {"User-Agent": "Boscdex-Fungi-Classifier/2.0"}

def download_and_save(url, dest_path):
    try:
        resp = requests.get(url, headers=HEADERS, timeout=12)
        if resp.status_code == 200:
            img = Image.open(io.BytesIO(resp.content)).convert("RGB")
            w, h = img.size
            if w < 120 or h < 120:
                return False
            min_dim = min(w, h)
            left = (w - min_dim) // 2
            top = (h - min_dim) // 2
            img = img.crop((left, top, left + min_dim, top + min_dim))
            img = img.resize((256, 256), Image.Resampling.LANCZOS)
            img.save(dest_path, "JPEG", quality=92)
            return True
    except Exception:
        pass
    return False

def count_valid_images(folder):
    if not folder.exists():
        return 0
    return len([p for p in folder.glob("*.*") if p.suffix.lower() in [".jpg", ".jpeg", ".png"]])

def top_up_species(s_id, taxon_id, target_count):
    folder = DATASET_DIR / s_id
    folder.mkdir(parents=True, exist_ok=True)
    existing = count_valid_images(folder)
    if existing >= target_count:
        return existing

    downloaded = existing
    file_idx = 1
    seen_urls = set()

    # Progressive geographical widening: Catalonia -> Spain -> Europe -> Worldwide
    for pid in [PLACE_CATALONIA, PLACE_SPAIN, PLACE_EUROPE, None]:
        if downloaded >= target_count:
            break
        for page in range(1, 6):
            if downloaded >= target_count:
                break
            try:
                kwargs = {
                    "taxon_id": taxon_id,
                    "iconic_taxa": "Fungi",  # Strictly enforce Kingdom Fungi
                    "quality_grade": "research",
                    "photos": True,
                    "per_page": 50,
                    "page": page,
                }
                if pid is not None:
                    kwargs["place_id"] = pid

                obs = get_observations(**kwargs)
                results = obs.get("results", [])
                if not results:
                    break

                urls = []
                for o in results:
                    for p in o.get("photos", []):
                        u = p.get("url", "").replace("square", "medium")
                        if u and u not in seen_urls:
                            seen_urls.add(u)
                            urls.append(u)

                for u in urls:
                    if downloaded >= target_count:
                        break
                    while (folder / f"inat_{file_idx:04d}.jpg").exists():
                        file_idx += 1
                    dest = folder / f"inat_{file_idx:04d}.jpg"
                    if download_and_save(u, dest):
                        downloaded += 1
                        file_idx += 1
                        time.sleep(0.04)
            except Exception:
                time.sleep(0.5)
                break
    return count_valid_images(folder)

if MIN_IMAGES_PER_SPECIES > 0:
    print(f"⚖️ Balancing dataset: Ensuring all {len(SPECIES_CATALOG)} fungal species have at least {MIN_IMAGES_PER_SPECIES} images...")
    for s_id, t_id, name in tqdm(SPECIES_CATALOG, desc="Balancing Taxa"):
        count = top_up_species(s_id, t_id, target_count=MIN_IMAGES_PER_SPECIES)
        print(f"  • {s_id.ljust(28)}: {count:3d} images ({name})")
        time.sleep(0.1)

    total = sum(count_valid_images(DATASET_DIR / s_id) for s_id, _, _ in SPECIES_CATALOG)
    print(f"✅ Dataset fully balanced! Total verified fungal images available: {total}")
else:
    print("Skipping iNaturalist download (MIN_IMAGES_PER_SPECIES=0).")


⚖️ Balancing dataset: Ensuring all 47 fungal species have at least 60 images...


Balancing Taxa:   0%|          | 0/47 [00:00<?, ?it/s]

  • lactarius_deliciosus        :  60 images (Rovelló / Pinetell)
  • boletus_edulis              :  60 images (Cep)
  • cantharellus_cibarius       :  60 images (Rossinyol)
  • amanita_caesarea            :  60 images (Ou de reig)
  • craterellus_cornucopioides  :  60 images (Trompeta de la mort)
  • macrolepiota_procera        :  60 images (Apagallums)
  • tricholoma_terreum          :  60 images (Fredolic)
  • calocybe_gambosa            :  60 images (Moixernó)
  • pleurotus_ostreatus         :  60 images (Gírgola)
  • hydnum_repandum             :  60 images (Llengua de bou)
  • hygrophorus_latitabundus    :  60 images (Llenega negra)
  • craterellus_lutescens       :  60 images (Camagroc)
  • suillus_luteus              :  60 images (Llenegot)
  • russula_cyanoxantha         :  60 images (Llora verda)
  • amanita_muscaria            :  60 images (Reig bord)
  • omphalotus_olearius         :  60 images (Gírgola d'olivera)
  • entoloma_sinuatum           :  60 images (Pèrfid)
  • am

In [6]:
# Step 6: Stratified Split, Safety-Weighted Loss, Label Smoothing & Frozen-BatchNorm Fine-Tuning
import random
import shutil
import tensorflow as tf
import numpy as np

print(f"TensorFlow Version: {tf.__version__}")
gpus = tf.config.list_physical_devices("GPU")
print(f"GPU Acceleration:    {'✅ Available (' + gpus[0].name + ')' if gpus else '⚠️ CPU Only'}")

# 1. Build Deterministic Stratified Train (85%) / Validation (15%) Split per Species
if SPLIT_DIR.exists():
    shutil.rmtree(SPLIT_DIR)
train_dir = SPLIT_DIR / "train"
val_dir = SPLIT_DIR / "val"
train_dir.mkdir(parents=True, exist_ok=True)
val_dir.mkdir(parents=True, exist_ok=True)

rng = random.Random(1337)
species_train_counts = {}
total_train_files = 0
total_val_files = 0

for s_id, _, _ in sorted(SPECIES_CATALOG, key=lambda x: x[0]):
    src_folder = DATASET_DIR / s_id
    if not src_folder.exists():
        continue
    valid_files = sorted([
        p for p in src_folder.glob("*.*")
        if p.suffix.lower() in [".jpg", ".jpeg", ".png"]
    ])
    if not valid_files:
        continue

    # Separate curated user field photos from iNaturalist top-up photos so both are stratified
    field_files = [p for p in valid_files if not p.name.startswith("inat_")]
    inat_files = [p for p in valid_files if p.name.startswith("inat_")]
    rng.shuffle(field_files)
    rng.shuffle(inat_files)

    def split_list(items, val_ratio=0.15):
        if len(items) <= 1:
            return items, []
        n_val = max(1, int(round(len(items) * val_ratio)))
        return items[n_val:], items[:n_val]

    f_train, f_val = split_list(field_files, 0.15)
    i_train, i_val = split_list(inat_files, 0.15)
    s_train = f_train + i_train
    s_val = f_val + i_val
    if not s_val and len(s_train) > 1:
        s_val = [s_train.pop()]

    (train_dir / s_id).mkdir(parents=True, exist_ok=True)
    (val_dir / s_id).mkdir(parents=True, exist_ok=True)

    for p in s_train:
        shutil.copy2(p, train_dir / s_id / p.name)
    for p in s_val:
        shutil.copy2(p, val_dir / s_id / p.name)

    species_train_counts[s_id] = len(s_train)
    total_train_files += len(s_train)
    total_val_files += len(s_val)

print(f"✓ Stratified split complete: {total_train_files} training images, {total_val_files} validation images across {len(species_train_counts)} classes.")

IMG_SIZE = (224, 224)
BATCH_SIZE = 32

train_ds = tf.keras.utils.image_dataset_from_directory(
    train_dir,
    seed=1337,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="categorical",
    crop_to_aspect_ratio=True,
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    val_dir,
    seed=1337,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="categorical",
    crop_to_aspect_ratio=True,
)

class_names = train_ds.class_names
num_classes = len(class_names)
print(f"Classes ({num_classes}): {class_names}")

# 2. Compute Balanced Class Weights + 1.20x Safety Priority Multiplier for Lethal & High-Risk Taxa
HIGH_PRIORITY_SAFETY_TAXA = {
    "amanita_phalloides",
    "amanita_verna",
    "amanita_virosa",
    "cortinarius_orellanus",
    "galerina_marginata",
    "gyromitra_esculenta",
    "amanita_pantherina",
    "tricholoma_pardinum",
    "entoloma_sinuatum",
    "rubroboletus_satanas",
    "omphalotus_olearius",
    "lactarius_torminosus",
    "agaricus_xanthodermus",
    "tricholoma_equestre",
}

class_weights = {}
for idx, c_name in enumerate(class_names):
    count = max(1, species_train_counts.get(c_name, 1))
    base_w = total_train_files / (num_classes * count)
    # Clamp extreme weights and apply safety boost
    w = float(np.clip(base_w, 0.6, 2.5))
    if c_name in HIGH_PRIORITY_SAFETY_TAXA:
        w *= 1.20
    class_weights[idx] = w

# 3. Mycological Data Augmentation (Macro Zoom + Canopy Lighting Simulation)
AUTOTUNE = tf.data.AUTOTUNE
data_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal"),
    tf.keras.layers.RandomRotation(0.20),
    tf.keras.layers.RandomZoom(height_factor=(-0.30, 0.10), width_factor=(-0.30, 0.10)),
    tf.keras.layers.RandomTranslation(0.08, 0.08),
    tf.keras.layers.RandomContrast(0.25),
    tf.keras.layers.RandomBrightness(0.25),
])

train_ds = train_ds.map(
    lambda x, y: (data_augmentation(x, training=True), y),
    num_parallel_calls=AUTOTUNE
).prefetch(buffer_size=AUTOTUNE)

val_ds = val_ds.prefetch(buffer_size=AUTOTUNE)

# 4. Base MobileNetV2 pre-trained on ImageNet
base_model = tf.keras.applications.MobileNetV2(
    input_shape=(224, 224, 3),
    include_top=False,
    weights="imagenet",
)
base_model.trainable = False

# Architecture with Named Layers for Decoupled Extraction
inputs = tf.keras.Input(shape=(224, 224, 3), name="input_image")
x = tf.keras.applications.mobilenet_v2.preprocess_input(inputs)
x = base_model(x, training=False)
gap = tf.keras.layers.GlobalAveragePooling2D(name="gap_features")(x)
dropout = tf.keras.layers.Dropout(0.30, name="dropout")(gap)
head = tf.keras.layers.Dense(
    num_classes,
    kernel_regularizer=tf.keras.regularizers.l2(1e-4),
    name="classifier_head",
)(dropout)
outputs = tf.keras.layers.Softmax(name="softmax")(head)

model = tf.keras.Model(inputs=inputs, outputs=outputs, name="boscdex_classifier")

loss_fn = tf.keras.losses.CategoricalCrossentropy(label_smoothing=0.10)
top3_metric = tf.keras.metrics.TopKCategoricalAccuracy(k=3, name="top3_acc")

reduce_lr_p1 = tf.keras.callbacks.ReduceLROnPlateau(
    monitor="val_loss", factor=0.5, patience=2, min_lr=5e-5, verbose=1
)

model.compile(
    optimizer=tf.keras.optimizers.AdamW(learning_rate=1e-3, weight_decay=1e-4),
    loss=loss_fn,
    metrics=["accuracy", top3_metric],
)

print()
print("🚀 Phase 1: Training top classification head (15 epochs with label smoothing & safety weights)...")
model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=15,
    class_weight=class_weights,
    callbacks=[reduce_lr_p1],
)

print()
print("🚀 Phase 2: Fine-tuning upper MobileNetV2 blocks (up to 25 epochs, BatchNorm frozen)...")
base_model.trainable = True
for i, layer in enumerate(base_model.layers):
    if i < 90 or isinstance(layer, tf.keras.layers.BatchNormalization):
        layer.trainable = False

reduce_lr_p2 = tf.keras.callbacks.ReduceLROnPlateau(
    monitor="val_loss", factor=0.5, patience=2, min_lr=1e-6, verbose=1
)
early_stop = tf.keras.callbacks.EarlyStopping(
    monitor="val_accuracy", mode="max", patience=6, restore_best_weights=True, verbose=1
)

model.compile(
    optimizer=tf.keras.optimizers.AdamW(learning_rate=3e-5, weight_decay=1e-4),
    loss=loss_fn,
    metrics=["accuracy", top3_metric],
)
model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=25,
    class_weight=class_weights,
    callbacks=[reduce_lr_p2, early_stop],
)

val_loss, val_acc, val_top3 = model.evaluate(val_ds)
print()
print(f"📊 Final Validation Top-1 Accuracy: {val_acc * 100:.2f}% | Top-3 Accuracy: {val_top3 * 100:.2f}%")


TensorFlow Version: 2.21.0
GPU Acceleration:    ✅ Available (/physical_device:GPU:0)
✓ Stratified split complete: 2392 training images, 428 validation images across 47 classes.
Found 2392 files belonging to 47 classes.
Found 428 files belonging to 47 classes.
Classes (47): ['agaricus_campestris', 'agaricus_xanthodermus', 'amanita_caesarea', 'amanita_muscaria', 'amanita_pantherina', 'amanita_phalloides', 'amanita_verna', 'amanita_virosa', 'boletus_aereus', 'boletus_edulis', 'calocybe_gambosa', 'cantharellus_cibarius', 'chroogomphus_rutilus', 'coprinus_comatus', 'cortinarius_orellanus', 'craterellus_cornucopioides', 'craterellus_lutescens', 'entoloma_sinuatum', 'galerina_marginata', 'gyromitra_esculenta', 'hydnum_repandum', 'hygrophorus_eburneus', 'hygrophorus_latitabundus', 'hygrophorus_personii', 'hygrophorus_russula', 'lactarius_chrysorrheus', 'lactarius_deliciosus', 'lactarius_sanguifluus', 'lactarius_semisanguifluus', 'lactarius_torminosus', 'lepista_nuda', 'macrolepiota_procera', '

In [7]:
# Step 7: Export Decoupled Model Assets for Boscdex Flutter App
import struct
import json

EXPORT_DIR = Path("./export")
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

# 1. Export labels.txt
labels_path = EXPORT_DIR / "mushroom_labels.txt"
with open(labels_path, "w", encoding="utf-8") as f:
    for name in class_names:
        f.write(f"{name}\n")
print(f"✓ Exported {len(class_names)} classes to: {labels_path}")

# 2. Export GAP Feature Extractor (Input: [1, 224, 224, 3], Output: [1, 1280] Float32)
# This matches TfliteClassifierService in Flutter exactly!
gap_model = tf.keras.Model(inputs=model.input, outputs=model.get_layer("gap_features").output)
converter = tf.lite.TFLiteConverter.from_keras_model(gap_model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
tflite_gap = converter.convert()

tflite_path = EXPORT_DIR / "mushroom_classifier.tflite"
with open(tflite_path, "wb") as f:
    f.write(tflite_gap)
print(f"✓ Exported GAP feature extractor: {tflite_path} ({tflite_path.stat().st_size / (1024*1024):.2f} MB)")

# 3. Export Portable Binary Classification Head (classifier_head.bin)
# Format: N float32 biases followed by N x 1280 float32 weights (little-endian)
dense_layer = model.get_layer("classifier_head")
weights, biases = dense_layer.get_weights()  # weights shape: (1280, N), biases shape: (N,)

head_bin_path = EXPORT_DIR / "mushroom_classifier_head.bin"
with open(head_bin_path, "wb") as f:
    # Write biases (len(class_names) floats)
    for b in biases:
        f.write(struct.pack("<f", float(b)))
    # Write weights (len(class_names) classes x 1280 features)
    for c in range(len(class_names)):
        for feature_idx in range(1280):
            w = weights[feature_idx, c]
            f.write(struct.pack("<f", float(w)))

print(f"✓ Exported binary classification head: {head_bin_path} ({head_bin_path.stat().st_size / 1024:.1f} KB)")

# 4. Also export full end-to-end model for reference
full_converter = tf.lite.TFLiteConverter.from_keras_model(model)
full_tflite = full_converter.convert()
full_path = EXPORT_DIR / "mushroom_classifier_full.tflite"
with open(full_path, "wb") as f:
    f.write(full_tflite)
print(f"✓ Exported full reference model: {full_path} ({full_path.stat().st_size / (1024*1024):.2f} MB)")


✓ Exported 47 classes to: export/mushroom_labels.txt
Saved artifact at '/tmp/tmpribmg8ue'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 224, 224, 3), dtype=tf.float32, name='input_image')
Output Type:
  TensorSpec(shape=(None, 1280), dtype=tf.float32, name=None)
Captures:
  136228983732560: TensorSpec(shape=(), dtype=tf.resource, name=None)
  136228982424016: TensorSpec(shape=(), dtype=tf.resource, name=None)
  136228982424208: TensorSpec(shape=(), dtype=tf.resource, name=None)
  136228983733712: TensorSpec(shape=(), dtype=tf.resource, name=None)
  136228982423824: TensorSpec(shape=(), dtype=tf.resource, name=None)
  136228982424592: TensorSpec(shape=(), dtype=tf.resource, name=None)
  136228982425360: TensorSpec(shape=(), dtype=tf.resource, name=None)
  136228982425168: TensorSpec(shape=(), dtype=tf.resource, name=None)
  136228982425552: TensorSpec(shape=(), dtype=tf.resource, name=None)
  136228982423632: TensorSpec(s

In [8]:
# Step 8: Export & Download Model Files
import shutil

# 1. If running locally or in VS Code with workspace root accessible, auto-copy to assets/models/
local_assets_targets = [Path("assets/models"), Path("../assets/models")]
for target in local_assets_targets:
    if target.exists() and target.is_dir():
        for f in [tflite_path, head_bin_path, labels_path]:
            shutil.copy2(f, target / f.name)
        print(f"✅ Automatically copied trained models to local workspace: {target.resolve()}/")
        break

# 2. If Google Drive is mounted, also back up to Google Drive
drive_models_dir = Path("/content/drive/MyDrive/boscdex_mushroom_models")
if Path("/content/drive/MyDrive").exists():
    drive_models_dir.mkdir(parents=True, exist_ok=True)
    for f in [tflite_path, head_bin_path, labels_path]:
        shutil.copy2(f, drive_models_dir / f.name)
    print(f"💾 Saved a permanent copy to Google Drive: {drive_models_dir}/")

# 3. Attempt direct browser download (works in web Colab)
try:
    from google.colab import files
    print("Triggering browser download of Boscdex model assets...")
    files.download(str(tflite_path))
    files.download(str(head_bin_path))
    files.download(str(labels_path))
except Exception as e:
    print(f"Browser download skipped/not supported: {e}")

print()
print("🎉 Done! Copy these 3 files into your local project:")
print("   1. assets/models/mushroom_classifier.tflite")
print("   2. assets/models/mushroom_classifier_head.bin")
print("   3. assets/models/mushroom_labels.txt")
print()
print("Then verify with:")
print("   flutter test test/unit_tests/domain/mushroom_model_real_images_test.dart")


💾 Saved a permanent copy to Google Drive: /content/drive/MyDrive/boscdex_mushroom_models/
Triggering browser download of Boscdex model assets...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


🎉 Done! Copy these 3 files into your local project:
   1. assets/models/mushroom_classifier.tflite
   2. assets/models/mushroom_classifier_head.bin
   3. assets/models/mushroom_labels.txt

Then verify with:
   flutter test test/unit_tests/domain/mushroom_model_real_images_test.dart
